# Figure 2B — SVs absent from HPRC/HGSVC3, split by Phase 1 status

Cumulative Phase 2 SVs (DEL / INS, 50 bp–10 kb, as in Table 2) that no HPRC/HGSVC3
control carries, along the Figure 2B participant order. Three classes:

- `in_phase1`: Truvari-matched to a Phase 1 site
- `new_singleton`: not in Phase 1, one carrier among participants
- `new_recurrent`: not in Phase 1, ≥2 carriers among participants

1. Stage scripts: `gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/"`
   (needs `panel_b_novelty_discovery.py` and `sv_site_utils.py`).
2. Upload `fig2_discovery_order.tsv` (written by `fig2_options.write_discovery_order`
   in the manuscript repo) to `$WORKSPACE_BUCKET/metadata/`.
3. Run top to bottom; after the symbolic-allele cell, adjust `PCTSEQ` if needed.

## Outputs

- `summaries/fig2/fig2_panel_b_discovery.tsv`: cumulative counts every 25 participants, no sample ids
- `summaries/fig2/fig2_panel_b_discovery.summary.json`: skip counts, class totals, Truvari totals

Copy both into `aou-lr-phase-2-manuscript/data/`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "panel_b_novelty_discovery.py").is_file() or (_d / "terra_notebook.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
if _scripts and str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))
print("scripts:", _scripts)
assert (_scripts / "panel_b_novelty_discovery.py").is_file(), "panel_b_novelty_discovery.py not staged"

In [ ]:
import json

BUCKET = "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09"
OUT_DIR = Path("summaries/fig2").resolve()
WORK = Path("work/fig2b").resolve()

# aou_lr_svs data table: AnnotateSvCallset outputs behind Table 2.
_ANN = f"{BUCKET}/submissions/016df050-9259-484d-8009-9cabc8cc5cbf/AnnotateSvCallset"
P1_BCF = f"{BUCKET}/scratch/kvg/phase1/concat_annotated.sens_07.bcf"
P1_CARRIERS = f"{_ANN}/898539c2-c34b-4e15-b2d6-69a365912ade/call-MainPart/aou_lr_phase1.main.carriers.tsv"
P2_BCF = "gs://prod-drc-broad/longreads/demo_group/integrated_sv_callset/v3_main.bcf"
PHASE2_SITES = f"{_ANN}/a986e347-3cf8-445c-ada2-96b0737f77dd/call-IntegrateAndSummarize/aou_lr_phase2.unified.sites.tsv"
PHASE2_CARRIERS = f"{_ANN}/a986e347-3cf8-445c-ada2-96b0737f77dd/call-MainPart/aou_lr_phase2.main.carriers.tsv"
ORDER = f"{BUCKET}/metadata/fig2_discovery_order.tsv"
CONTROLS = str(_scripts.parent / "tractor_mix/reference_controls/control_sample_metadata.tsv")
JOBS = os.cpu_count() or 4
STEP = 25

if not Path(CONTROLS).is_file():
    raise FileNotFoundError(f"{CONTROLS} missing; sync the repo first")
for d in (OUT_DIR, WORK / "sites", WORK / "bench", WORK / "logs"):
    d.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "P1_BCF": P1_BCF, "P1_CARRIERS": P1_CARRIERS, "P2_BCF": P2_BCF, "PHASE2_SITES": PHASE2_SITES,
    "PHASE2_CARRIERS": PHASE2_CARRIERS, "ORDER": ORDER, "CONTROLS": CONTROLS,
    "JOBS": JOBS, "WORK": str(WORK), "OUT_DIR": str(OUT_DIR),
}, indent=2))

In [ ]:
import shutil

try:
    import truvari  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "truvari"])
TRUVARI = [sys.executable, "-m", "truvari"]
print(subprocess.run(TRUVARI + ["version"], capture_output=True, text=True).stdout.strip())
assert shutil.which("bcftools") and shutil.which("tabix"), "bcftools / tabix not on PATH"


def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True, text=True, **kw)


def stage(uri: str) -> Path:
    """Local path for a gs:// URI (copied once into WORK) or an existing local file."""
    if not uri.startswith("gs://"):
        return Path(uri).resolve()
    dest = WORK / Path(uri).name
    if not dest.exists():
        sh(["gsutil", "-q", "cp", uri, str(dest)])
    return dest

In [ ]:
# Stage inputs, then sites-only (no genotypes), indexed copies of both SV callsets.
sites_tsv = stage(PHASE2_SITES)
carriers_tsv = stage(PHASE2_CARRIERS)
order_tsv = stage(ORDER)
controls_tsv = stage(CONTROLS)

import csv

# Phase 1 sites carried only by the 47 Phase 1 reference controls are not Phase 1 participant variation.
controls = set()
with open(controls_tsv) as fh:
    for r in csv.DictReader(fh, delimiter="\t"):
        if r.get("is_reference_control", "").strip().lower() in {"1", "true", "t", "yes", "y"}:
            controls.add(r["research_id"].strip())
p1_keep = WORK / "phase1_participant_ids.txt"
n_all = n_keep = 0
with open(stage(P1_CARRIERS)) as fh, p1_keep.open("w") as out:
    for r in csv.DictReader(fh, delimiter="\t"):
        n_all += 1
        if any(s and s not in controls for s in (r["carriers"] or "").split(",")):
            out.write(r["id"] + "\n")
            n_keep += 1
print(f"Phase 1 sites with a participant carrier: {n_keep:,} / {n_all:,}")

sites_vcf = {}
for tag, uri in (("p1", P1_BCF), ("p2", P2_BCF)):
    out = WORK / "sites" / f"{tag}.vcf.gz"
    if not out.exists():
        src = stage(uri)
        keep = ["-i", f"ID=@{p1_keep}"] if tag == "p1" else []
        sh(["bcftools", "view", "-G", *keep, "-Oz", "-o", str(out), str(src)])
        sh(["tabix", "-f", "-p", "vcf", str(out)])
    sites_vcf[tag] = out

for tag, path in sites_vcf.items():
    n = sh(["bcftools", "index", "-n", str(path)], capture_output=True).stdout.strip()
    alts = sh(f"bcftools query -f '%ALT\n' {path} | grep -c '^<' || true", capture_output=True).stdout.strip()
    print(f"{tag}: {int(n):,} records, {int(alts or 0):,} symbolic ALTs")

## Matching thresholds

Symbolic ALTs (`<DEL>`, `<INS>`) carry no sequence, so they cannot pass a sequence-similarity
check. If either callset has many, set `PCTSEQ = 0` (size and position only); otherwise keep
`0.7`. Ideally these match the Truvari merge used inside the Phase 2 pipeline.

In [ ]:
PCTSEQ = 0.7  # set to 0 if either callset has many symbolic ALTs
PCTSIZE = 0.7
REFDIST = 500
print({"PCTSEQ": PCTSEQ, "PCTSIZE": PCTSIZE, "REFDIST": REFDIST})

In [ ]:
# Truvari per chromosome, in parallel: Phase 1 as base, Phase 2 as comparison.
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

contigs = [
    c for c in sh(["tabix", "-l", str(sites_vcf["p2"])], capture_output=True).stdout.split()
    if re.fullmatch(r"(chr)?([0-9]+|X|Y)", c)
]


def bench_chrom(c: str) -> str:
    parts = {}
    for tag in ("p1", "p2"):
        out = WORK / "sites" / f"{tag}.{c}.vcf.gz"
        subprocess.run(["bcftools", "view", "-r", c, "-Oz", "-o", str(out), str(sites_vcf[tag])], check=True)
        subprocess.run(["tabix", "-f", "-p", "vcf", str(out)], check=True)
        parts[tag] = out
    odir = WORK / "bench" / c
    if odir.exists():
        shutil.rmtree(odir)
    with open(WORK / "logs" / f"bench.{c}.log", "w") as log:
        subprocess.run(
            TRUVARI + [
                "bench", "-b", str(parts["p1"]), "-c", str(parts["p2"]), "-o", str(odir),
                "--sizemin", "50", "--sizemax", "10000",
                "--pctseq", str(PCTSEQ), "--pctsize", str(PCTSIZE), "--refdist", str(REFDIST),
                "--pick", "multi",
            ],
            check=True, stdout=log, stderr=subprocess.STDOUT,
        )
    return c


with ThreadPoolExecutor(max_workers=JOBS) as pool:
    futures = {pool.submit(bench_chrom, c): c for c in contigs}
    for f in as_completed(futures):
        print("done", f.result())

In [ ]:
# Phase 2 sites matched to Phase 1, plus Truvari totals.
matched_tsv = WORK / "phase1_matched.tsv"
tp = fp = fn = tp_base = 0
with matched_tsv.open("w") as out:
    for c in contigs:
        bdir = WORK / "bench" / c
        out.write(sh(["bcftools", "query", "-f", "%CHROM\t%POS\t%ID\n", str(bdir / "tp-comp.vcf.gz")], capture_output=True).stdout)
        s = json.loads((bdir / "summary.json").read_text())
        tp += s["TP-comp"]; fp += s["FP"]; fn += s["FN"]; tp_base += s["TP-base"]
truvari_totals = {
    "phase2_matched": tp, "phase2_unmatched": fp,
    "phase1_matched": tp_base, "phase1_unmatched": fn,
    "phase1_recovered_frac": tp_base / max(tp_base + fn, 1),
    "phase2_absent_from_phase1_frac": fp / max(tp + fp, 1),
    "params": {"pctseq": PCTSEQ, "pctsize": PCTSIZE, "refdist": REFDIST, "sizemin": 50, "sizemax": 10000},
}
print(json.dumps(truvari_totals, indent=2))

In [ ]:
out_tsv = OUT_DIR / "fig2_panel_b_discovery.tsv"
out_sum = OUT_DIR / "fig2_panel_b_discovery.summary.json"
sh([
    sys.executable, str(_scripts / "panel_b_novelty_discovery.py"),
    "--sites", str(sites_tsv), "--carriers", str(carriers_tsv),
    "--order", str(order_tsv), "--controls", str(controls_tsv),
    "--phase1-matched", str(matched_tsv), "--step", str(STEP),
    "--out", str(out_tsv), "--out-summary", str(out_sum),
])
summary = json.loads(out_sum.read_text())
summary["truvari"] = truvari_totals
out_sum.write_text(json.dumps(summary, indent=2) + "\n")
c = summary["counts"]
print(f"participants {summary['n_participants']:,}  controls {summary['n_controls']}")
print("skipped: ambiguous id", c.get("skip_ambiguous_id", 0), " suppressed duplicate", c.get("skip_suppressed_duplicate", 0),
      " no participant carrier", c.get("skip_no_participant_carrier", 0))
for sv in ("INS", "DEL"):
    print(sv, {k: c.get(f"{k}_{sv}", 0) for k in ("in_controls", "in_phase1", "new_singleton", "new_recurrent")})

In [ ]:
sh(["gsutil", "cp", str(out_tsv), str(out_sum), f"{BUCKET}/figures/"])